In [2]:
import pandas as pd
import glob
import re
import os

In [14]:
df_uf = pd.read_csv(r'C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv final\leitos_populacao_uf_2023_2025.csv')

In [10]:
def carregar_obitos_sih(caminho):
    # Lê as 3 primeiras linhas para extrair o período
    with open(caminho, encoding="latin1") as f:
        linhas = [next(f) for _ in range(3)]

    periodo_str = linhas[2]

    # Exemplo esperado: "Período:Dez/2023"
    mes_ano = re.search(r"(\w{3})/(\d{4})", periodo_str)

    if mes_ano is None:
        raise ValueError(
            f"Não foi possível identificar o período no arquivo: {caminho}"
        )

    mes_abrev = mes_ano.group(1)
    ano = int(mes_ano.group(2))

    meses = {
        "Jan": 1, "Fev": 2, "Mar": 3, "Abr": 4,
        "Mai": 5, "Jun": 6, "Jul": 7, "Ago": 8,
        "Set": 9, "Out": 10, "Nov": 11, "Dez": 12
    }

    mes = meses[mes_abrev]

    # Lê os dados
    df = pd.read_csv(
        caminho,
        encoding="latin1",
        sep=";",
        skiprows=3
    )

    df.columns = ["UF_COD_NOME", "OBITOS", "TOTAL"]

    # Remove a coluna TOTAL
    df = df.drop(columns=["TOTAL"])

    # Extrai código IBGE da UF
    df["CO_UF_IBGE"] = df["UF_COD_NOME"].str.extract(r"^(\d+)")

    # Extrai nome da UF
    df["NOME_UF"] = df["UF_COD_NOME"].str.replace(
        r"^\d+\s*",
        "",
        regex=True
    )

    df = df.drop(columns=["UF_COD_NOME"])

    # Adiciona ano e mês
    df["ANO_REF"] = ano
    df["MES_REF"] = mes

    return df


arquivos = glob.glob(
    "../data/csv a ser tratado/obitos/*/*.csv"
)

print(f"Total de arquivos encontrados: {len(arquivos)}")


# ============================================================
# JUNTA TODOS OS ARQUIVOS
# ============================================================

df_obitos = pd.concat(
    [carregar_obitos_sih(a) for a in arquivos],
    ignore_index=True
)

# Remove linhas de rodapé que não possuem código IBGE
df_obitos = df_obitos.dropna(
    subset=["CO_UF_IBGE"]
)


# ============================================================
# RESULTADO
# ============================================================

print("Formato:", df_obitos.shape)
print(df_obitos.head(15))

Total de arquivos encontrados: 36


Formato: (972, 5)
    OBITOS CO_UF_IBGE              NOME_UF  ANO_REF  MES_REF
0    266.0         11             Rondônia     2023       12
1     99.0         12                 Acre     2023       12
2    357.0         13             Amazonas     2023       12
3    102.0         14              Roraima     2023       12
4   1121.0         15                 Pará     2023       12
5    140.0         16                Amapá     2023       12
6    280.0         17            Tocantins     2023       12
7   1137.0         21             Maranhão     2023       12
8    778.0         22                Piauí     2023       12
9   1806.0         23                Ceará     2023       12
10   550.0         24  Rio Grande do Norte     2023       12
11   807.0         25              Paraíba     2023       12
12  2267.0         26           Pernambuco     2023       12
13   697.0         27              Alagoas     2023       12
14   426.0         28              Sergipe     2023       12


In [12]:
# 1. Confirma que cada UF aparece exatamente 12x por ano (nenhum mês duplicado ou faltando)
print(df_obitos.groupby("ANO_REF")["MES_REF"].nunique())  # deve dar 12 pra cada ano

# 2. Confirma que não há duplicata de UF dentro do mesmo mês/ano
print("Duplicatas:", df_obitos.duplicated(subset=["NOME_UF", "ANO_REF", "MES_REF"]).sum())  # deve dar 0

ANO_REF
2023    12
2024    12
2025    12
Name: MES_REF, dtype: int64
Duplicatas: 0


In [15]:
# 1. Agregação ANUAL de óbitos (soma dos 12 meses) -> comparável em granularidade com df_uf
obitos_anual = df_obitos.groupby(["NOME_UF", "ANO_REF"]).agg(
    TOTAL_OBITOS=("OBITOS", "sum")
).reset_index()

# 2. Join com o dataset que já tínhamos (leitos + população + taxa)
df_uf_completo = df_uf.merge(obitos_anual, on=["NOME_UF", "ANO_REF"], how="left")

# 3. Métrica derivada: óbitos hospitalares por 10 mil habitantes (mesma lógica da taxa de leitos)
df_uf_completo["OBITOS_POR_10K_HAB"] = (df_uf_completo["TOTAL_OBITOS"] / df_uf_completo["POPULACAO"]) * 10000

print(df_uf_completo.shape)
print(df_uf_completo.head())

# 4. Salva os arquivos
os.makedirs("data/processed", exist_ok=True)

# Base mensal de óbitos (nível mais granular, pra análise de sazonalidade/tendência)
df_obitos.to_csv("data/processed/obitos_sih_mensal_uf_2023_2025.csv", index=False, encoding="utf-8")

# Dataset unificado final: leitos + população + óbitos, por UF/ano
df_uf_completo.to_csv("data/processed/leitos_populacao_obitos_uf_2023_2025.csv", index=False, encoding="utf-8")

print("Óbitos mensal:", df_obitos.shape)
print("Dataset unificado (UF/ano):", df_uf_completo.shape)

(81, 10)
   NOME_UF  ANO_REF  TOTAL_LEITOS_EXISTENTES  TOTAL_LEITOS_SUS  \
0     Acre     2023                     1790              1547   
1     Acre     2024                     1793              1611   
2     Acre     2025                     1881              1694   
3  Alagoas     2023                     7338              5997   
4  Alagoas     2024                     7413              6062   

   QTD_ESTABELECIMENTOS  POPULACAO  CO_UF_IBGE  LEITOS_POR_10K_HAB  \
0                    34   876582.0          12           20.420223   
1                    34   880631.0          12           20.360401   
2                    34   884372.0          12           21.269330   
3                    98  3218607.0          27           22.798683   
4                    97  3220104.0          27           23.020996   

   TOTAL_OBITOS  OBITOS_POR_10K_HAB  
0        1786.0           20.374591  
1        1805.0           20.496667  
2        1791.0           20.251659  
3        7926.0      

In [17]:
print("Linhas com óbito faltante após o merge:", df_uf_completo["TOTAL_OBITOS"].isna().sum())

Linhas com óbito faltante após o merge: 0
